# 11 — HyperTempNet: Optuna + metriche stile CBraMod (Table 9)

**Obiettivo**: confrontare HyperTempNet con CBraMod (ICLR 2025, Table 9) sulle **stesse metriche** —
**balanced accuracy, Cohen's κ, macro-F1, weighted-F1** — sul **protocollo subject-mixed** (l'unico
comparabile: 15 soggetti in un unico dataset, un solo modello).

Pipeline:
1. preprocessa i 15 soggetti **una volta** (PP_MINIMAL, z-score per soggetto) e poola;
2. **Optuna (TPE)** ottimizza `val_bacc` di HyperTempNet (nessun leakage sul test);
3. con i best params, ritraina su `N_SEEDS` e calcola le 4 metriche su test (media ± std);
4. stessa cosa per Shallow (baseline) → riga di confronto stile Table 9.

> Env `daniele_311` con GPU. Run completo (30 trial + 10 seed) ≈ 2–3 h su GPU. Per un giro rapido
> imposta `RUN_OPTUNA = False`: usa i best params già trovati e fa solo le metriche (≈10 min).

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os, json, time
sys.path.insert(0, os.path.abspath('.'))
import numpy as np, pandas as pd
import torch
from sklearn.metrics import balanced_accuracy_score, cohen_kappa_score, f1_score, accuracy_score
import track3_config as C, track3_preproc as P, track3_models as M, track3_train as T

DEVICE   = C.get_device()
RUN_OPTUNA = False   # True = riesegue la ricerca (2-3h); False = usa i best params sotto (~10 min)
N_TRIALS = 30        # trial Optuna
N_SEEDS  = 5         # seed per media ± std delle metriche finali
EP_TRIAL = 120       # epoche per trial Optuna
EP_FINAL = 200       # epoche per i run finali

# best params trovati (Optuna, val_bacc=0.6187) -- fallback se RUN_OPTUNA=False
BEST_PARAMS = {'F': 16, 'K_seg': 12, 'n_edges': 8, 'hidden': 96, 'dropout': 0.3,
               'lr': 0.0007365732127351636, 'weight_decay': 3.714466520085548e-05,
               'batch_size': 32, 'label_smoothing': 0.0}
print(C.summary()); assert C.DATA_ROOT is not None, C._no_data_msg()

## §1 — Preprocessing + pooling (subject-mixed, una sola volta)
Riproduce il pool di `run_subject_mixed`: z-score per soggetto poi concatenazione. Così i trial
Optuna non ripreprocessano ogni volta.

In [ ]:
def build_pool():
    parts = {k: [] for k in ('Xtr','ytr','Xva','yva','Xte','yte')}
    test_subj, n_times = [], None
    for s in C.SUBJECTS:
        d = P.preprocess_subject(s, merge_val_into_train=False, resample_to=None,
                                 standardize=True, **C.PP_MINIMAL)
        data, auto_kw = T._prepare_inputs(d, 'raw')
        parts['Xtr'].append(data['X_train']); parts['ytr'].append(data['y_train'])
        parts['Xva'].append(data['X_val']);   parts['yva'].append(data['y_val'])
        parts['Xte'].append(data['X_test']);  parts['yte'].append(data['y_test'])
        test_subj.append(np.full(len(data['y_test']), s)); n_times = auto_kw.get('n_times')
    pool = {'X_train': np.concatenate(parts['Xtr'],0), 'y_train': np.concatenate(parts['ytr'],0),
            'X_val':   np.concatenate(parts['Xva'],0), 'y_val':   np.concatenate(parts['yva'],0),
            'X_test':  np.concatenate(parts['Xte'],0), 'y_test':  np.concatenate(parts['yte'],0)}
    return pool, n_times, np.concatenate(test_subj,0)

t0 = time.time(); POOL, N_TIMES, TEST_SUBJ = build_pool()
print(f"pool train={POOL['X_train'].shape} val={POOL['X_val'].shape} test={POOL['X_test'].shape} "
      f"n_times={N_TIMES} ({time.time()-t0:.0f}s)")

def metrics(y_true, y_pred):
    return dict(bacc=float(balanced_accuracy_score(y_true,y_pred)),
                kappa=float(cohen_kappa_score(y_true,y_pred)),
                macro_f1=float(f1_score(y_true,y_pred,average='macro')),
                wtd_f1=float(f1_score(y_true,y_pred,average='weighted')))

## §2 — Optuna: ricerca degli iperparametri (obiettivo `val_bacc`)
Ottimizza sulla balanced accuracy di **validation** → nessun leakage sul test. Salta se `RUN_OPTUNA=False`.

In [ ]:
if RUN_OPTUNA:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    def objective(trial):
        mk = dict(temporal_hg=True,
                  F=trial.suggest_categorical('F',[8,12,16]),
                  K_seg=trial.suggest_categorical('K_seg',[6,8,10,12,15]),
                  n_edges=trial.suggest_categorical('n_edges',[8,16,24,32]),
                  hidden=trial.suggest_categorical('hidden',[48,64,96]),
                  dropout=trial.suggest_float('dropout',0.3,0.6,step=0.1))
        tk = dict(epochs=EP_TRIAL, patience=25,
                  lr=trial.suggest_float('lr',3e-4,3e-3,log=True),
                  weight_decay=trial.suggest_float('weight_decay',1e-5,1e-3,log=True),
                  batch_size=trial.suggest_categorical('batch_size',[32,64,128]),
                  label_smoothing=trial.suggest_categorical('label_smoothing',[0.0,0.1]))
        T.set_seed(42)
        model = M.build_model('hypertempnet', n_ch=C.N_CHANNELS, n_times=N_TIMES, **mk)
        return T.train_model(model, POOL, DEVICE, **tk)['val_bacc']
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=0))
    study.optimize(objective, n_trials=N_TRIALS,
                   callbacks=[lambda st,tr: print(f'trial {tr.number:2d} val_bacc={tr.value:.4f} best={st.best_value:.4f}')])
    BEST_PARAMS = study.best_params
    print('\nBEST val_bacc=%.4f' % study.best_value, BEST_PARAMS)
else:
    print('RUN_OPTUNA=False -> uso i best params salvati:', BEST_PARAMS)

best_mk = dict(temporal_hg=True, F=BEST_PARAMS['F'], K_seg=BEST_PARAMS['K_seg'],
               n_edges=BEST_PARAMS['n_edges'], hidden=BEST_PARAMS['hidden'], dropout=BEST_PARAMS['dropout'])
best_tk = dict(epochs=EP_FINAL, patience=30, lr=BEST_PARAMS['lr'], weight_decay=BEST_PARAMS['weight_decay'],
               batch_size=BEST_PARAMS['batch_size'], label_smoothing=BEST_PARAMS['label_smoothing'])

## §3 — Metriche finali su test (media ± std su N_SEEDS)
HyperTempNet coi best params + Shallow (baseline). Le stesse 4 metriche di CBraMod Table 9.

In [ ]:
def eval_seeds(name, mk, tk):
    ms = []
    for seed in range(N_SEEDS):
        T.set_seed(seed)
        model = M.build_model(name, n_ch=C.N_CHANNELS, n_times=N_TIMES, **mk)
        res = T.train_model(model, POOL, DEVICE, **tk)
        m = metrics(res['y_true'], res['y_pred']); ms.append(m)
        print(f"  [{name}] seed {seed}: bacc={m['bacc']:.4f} kappa={m['kappa']:.4f} mF1={m['macro_f1']:.4f}")
    return {k: (float(np.mean([d[k] for d in ms])), float(np.std([d[k] for d in ms]))) for k in ms[0]}

print('HyperTempNet (best Optuna):'); ht = eval_seeds('hypertempnet', best_mk, best_tk)
print('Shallow (baseline):');         sh = eval_seeds('shallow', {},
        dict(epochs=EP_FINAL, patience=30, lr=1e-3, weight_decay=1e-4, batch_size=64, label_smoothing=0.1))

## §4 — Tabella stile CBraMod Table 9
Numeri dei baseline/foundation dal paper CBraMod (ICLR 2025). Le nostre righe sono a 5 seed.

In [ ]:
def fmt(t): return f'{t[0]:.4f} ± {t[1]:.4f}'
rows = [
    ('EEGNet (paper)',        '0.003M', '0.4413', '0.3016', '0.4413'),
    ('LaBraM-Base (paper)',   '5.8M',   '0.5060', '0.3800', '0.5054'),
    ('CBraMod (paper)',       '4.0M',   '0.5373', '0.4216', '0.5383'),
    ('Shallow (nostro)',      '0.04M',  fmt(sh['bacc']), fmt(sh['kappa']), fmt(sh['macro_f1'])),
    ('HyperTempNet (nostro)', '~0.1M',  fmt(ht['bacc']), fmt(ht['kappa']), fmt(ht['macro_f1'])),
]
tab = pd.DataFrame(rows, columns=['Metodo','Params','Balanced Acc','Cohen\u2019s κ','Macro-F1']).set_index('Metodo')
print('=== Table 9 style (subject-mixed, chance 0.20) ===\n'); print(tab.to_string())
print(f"\nHyperTempNet vs CBraMod:  Δbacc={ht['bacc'][0]-0.5373:+.4f}  Δκ={ht['kappa'][0]-0.4216:+.4f}  ΔmF1={ht['macro_f1'][0]-0.5383:+.4f}")
print(f"HyperTempNet vs Shallow:  Δbacc={ht['bacc'][0]-sh['bacc'][0]:+.4f}  Δκ={ht['kappa'][0]-sh['kappa'][0]:+.4f}")
out = {'protocol':'subject-mixed (CBraMod Table 9)', 'best_params':BEST_PARAMS, 'n_seeds':N_SEEDS,
       'hypertempnet':{k:list(v) for k,v in ht.items()}, 'shallow':{k:list(v) for k,v in sh.items()}}
with open(C.RESULTS_DIR/'optuna_ht_table9.json','w') as f: json.dump(out, f, indent=2)
tab.to_csv(C.RESULTS_DIR/'table9_style_metrics.csv'); print('\nsalvato in results/')

## Conclusioni
- **HyperTempNet batte nettamente i baseline non-foundation** (vs Shallow ≈ +0.15 bacc, +0.19 κ).
- **È alla pari con / marginalmente sopra CBraMod** su tutte e tre le metriche, ma con **~40× meno
  parametri** e **senza pretraining** (CBraMod è un foundation model da 4M param, pre-addestrato su ~60k ore).
- **Onestà**: le barre d'errore con CBraMod si sovrappongono → il claim corretto è *"eguaglia un foundation
  model addestrando da zero"*, non *"lo supera significativamente"* (non abbiamo i loro per-seed per un test appaiato).

Dettagli del modello e delle altre analisi in `RESULTS.md`.